In [0]:
dbutils.library.restartPython()

In [ ]:
import json
import ast

# Create widget to receive table_config from For Each
dbutils.widgets.text("table_config", "{}")
table_config_json = dbutils.widgets.get("table_config")

print(f"Raw parameter: {table_config_json[:200]}...")

# Parse the JSON
table_configs = json.loads(table_config_json)
print(f"\nProcessing table: {table_configs.get('source_table', 'UNKNOWN')}")

In [ ]:
from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig
)
from databricks.labs.lakebridge.reconcile.recon_config import Filters, Table, Transformation
from databricks.labs.lakebridge.reconcile.trigger_recon_service import TriggerReconService
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException
from databricks.sdk import WorkspaceClient
from databricks.labs.lakebridge import __version__
from dataclasses import dataclass

@dataclass
class TableRecon:
    source_schema: str
    target_catalog: str
    target_schema: str
    tables: list[Table]
    source_catalog: str | None = None

ws = WorkspaceClient(product="lakebridge", product_version=__version__)
print(" Libraries imported and workspace client initialized")

In [ ]:
try:
    reconcile_config = ReconcileConfig(
        data_source=table_configs["data_source"],
        report_type=table_configs["report_type"].lower(),
        secret_scope=table_configs["secret_scope"],  # ✓ Fixed: was "scope_name"

        database_config=DatabaseConfig(
            source_catalog=table_configs["source_catalog"],
            source_schema=table_configs["source_schema"],
            target_catalog=table_configs["target_catalog"],
            target_schema=table_configs["target_schema"]
        ),

        metadata_config=ReconcileMetadataConfig(
            catalog=table_configs["target_catalog"],
            schema="lakebridge_recon"
        ))

    # Handle filters dynamically based on filters_column_type from CSV
    filters_col = table_configs.get('filters_column')
    filters_col_type = table_configs.get('filters_column_type', '').lower().strip()
    table_filters = None
    
    if filters_col and str(filters_col).strip().lower() not in ['nan', 'none', '']:
        source_condition = table_configs.get('source_filters_condition', '')
        target_condition = table_configs.get('target_filters_condition', '')
        
        if filters_col_type == 'timestamp':
            # Timestamp/datetime columns - use special timestamp handling
            target_ts_expr = f"coalesce(try_to_timestamp({filters_col}), to_timestamp(regexp_replace(trim({filters_col}), ' +', ' '), 'MMM d yyyy h:mma'))"
            table_filters = Filters(
                source=f"{filters_col} {source_condition}",
                target=f"{target_ts_expr} {target_condition}"
            )
        elif filters_col_type == 'string':
            # String columns - use LOWER() for case-insensitive comparison
            table_filters = Filters(
                source=f"lower({filters_col}) {source_condition}",
                target=f"lower({filters_col}) {target_condition}"
            )
        elif filters_col_type == 'numeric':
            # Numeric columns - direct comparison
            table_filters = Filters(
                source=f"{filters_col} {source_condition}",
                target=f"{filters_col} {target_condition}"
            )
        else:
            # Default: direct comparison without transformation
            table_filters = Filters(
                source=f"{filters_col} {source_condition}",
                target=f"{filters_col} {target_condition}"
            )
        
        print(f"✓ Applied {filters_col_type or 'default'} filter on column: {filters_col}")

    # Parse transformations from CSV dynamically
    recon_transformations = []
    transformation_str = table_configs.get('transformation', '')
    
    if transformation_str and str(transformation_str).strip() not in ['', 'nan', 'None']:
        try:
            # Parse the transformation dictionary from CSV
            transformation_dict = ast.literal_eval(transformation_str)
            
            # Convert to Transformation objects
            for col_name, transforms in transformation_dict.items():
                recon_transformations.append(
                    Transformation(
                        column_name=col_name,
                        source=transforms['source'],
                        target=transforms['target']
                    )
                )
            print(f"✓ Loaded {len(recon_transformations)} transformations from CSV")
        except Exception as e:
            print(f"⚠ Warning: Could not parse transformations: {e}")

    table_recon = TableRecon(
        source_schema=table_configs["source_schema"],
        target_catalog=table_configs["target_catalog"],
        target_schema=table_configs["target_schema"],
        tables=[
            Table(
                source_name=table_configs["source_table"],
                target_name=table_configs["target_table"],  # ✓ Matches the key from config notebook
                join_columns=table_configs["join_columns"],  # ✓ Fixed: already a list, no need to split
                filters=table_filters,
                transformations=recon_transformations if recon_transformations else None
            )
        ]
    )

    result = TriggerReconService.trigger_recon(
        ws=ws,
        spark=spark,
        table_recon=table_recon,
        reconcile_config=reconcile_config
    )
    print(f"✓ Recon Success for {table_configs['source_table']} | Recon ID: {result.recon_id}")
except Exception as e:
    print(f"✗ Recon Failed: {e}")
    raise